In [2]:
#!/usr/bin/env python
"""
OH radical (7e,NORB)/cc-pVDZ PES scan — STATE-RESOLVED AMPLITUDE VERSION

Stores α/β marginals for EACH target state (D0, D1, D2, Q1) for:
  - Reference (FCI)
  - Ext-SQD
  - CIPSI

States: D0, D1, D2, Q1 (3 doublets + 1 quartet)

New wf_amps structure:
    wf_amps[state_label] = {
        "ref_a": [n_alpha floats],   # α-string marginals on full space
        "ref_b": [n_beta floats],    # β-string marginals
        "ext_a": [n_alpha floats],   # (0-padded where subspace missing)
        "ext_b": [n_beta floats],
        "cipsi_a": [n_alpha floats],
        "cipsi_b": [n_beta floats],
    }
    wf_amps_meta = {
        "n_alpha_full": int,
        "n_beta_full": int,
        "states_stored": [...],
    }
"""

# %% Cell 1: Imports
import os, time, json, numpy as np

from pyscf import gto, scf, cc, mcscf, ao2mo, fci, lib
from pyscf.fci import selected_ci, cistring
from qiskit.circuit import QuantumCircuit
from qiskit_addon_sqd.fermion import (
    enlarge_batch_from_transitions,
    bitstring_matrix_to_ci_strs,
    recover_configurations,
    subsample,
    postselect_by_hamming_right_and_left,
    solve_fermion,
)
import ffsim
import warnings
warnings.filterwarnings("ignore")

# ====== MULTICORE SETTING ======
N_THREADS = int(os.environ.get("OMP_NUM_THREADS", 24))
lib.num_threads(N_THREADS)
print(f"PySCF threads: {N_THREADS}")
# ================================

# %% Cell 2: Core functions

NORB = 18
NELEC = (4, 3)  # open-shell doublet
NCORE = 1

TARGET_STATES = ["D0", "D1", "D2", "Q1"]
MAX_DOUBLET = 3
MAX_QUARTET = 1

def safe_array(x):
    return np.nan_to_num(np.asarray(x, float))


def det_strings(norb, nelec):
    na, nb = nelec
    return (np.array(cistring.make_strings(range(norb), na), np.int64),
            np.array(cistring.make_strings(range(norb), nb), np.int64))

def reverse_bits(s, norb):
    """Reverse bit order within norb bits (fix BSM↔cistring convention)."""
    result = 0
    for i in range(norb):
        if (s >> i) & 1:
            result |= (1 << (norb - 1 - i))
    return result

def fix_ci_strs(strs, norb):
    """Fix CI strings from bitstring_matrix_to_ci_strs to match PySCF convention."""
    return np.array(sorted(set(reverse_bits(int(s), norb) for s in strs)), dtype=np.int64)

def classify_spin(s2):
    if abs(s2 - 0.75) < 0.4: return "doublet"
    if abs(s2 - 3.75) < 0.5: return "quartet"
    return "other"

def kernel_safe(myci, h1, eri, norb, nelec, ci_strs, nroots, loose=False):
    try:
        if loose:
            myci.conv_tol = GROW_CONV_TOL; myci.max_cycle = GROW_MAX_CYCLE
        else:
            myci.conv_tol = 1e-10; myci.max_cycle = 200
        e, c = selected_ci.kernel_fixed_space(
            myci, h1, eri, norb, nelec, ci_strs=ci_strs, nroots=nroots)
        if np.isscalar(e): return [float(e)], [c]
        return [float(x) for x in e], list(c)
    except:
        return [], []

def get_alpha_coeffs(c_full, strsa, basis, ci_strs):
    c = np.asarray(c_full)
    coeffs = np.zeros(len(basis))
    if c.ndim == 2:
        sm = {int(s): i for i, s in enumerate(ci_strs[0])}
        for bi, bs in enumerate(basis):
            ia = sm.get(int(bs))
            if ia is not None and ia < c.shape[0]:
                coeffs[bi] = np.max(np.abs(c[ia, :]))
    return coeffs

def get_beta_coeffs(c_full, strsb, basis_b, ci_strs):
    c = np.asarray(c_full)
    coeffs = np.zeros(len(basis_b))
    if c.ndim == 2:
        sm = {int(s): i for i, s in enumerate(ci_strs[1])}
        for bi, bs in enumerate(basis_b):
            ib = sm.get(int(bs))
            if ib is not None and ib < c.shape[1]:
                coeffs[bi] = np.max(np.abs(c[:, ib]))
    return coeffs

def _compute_diag(s, norb, h1, eri):
    """Diagonal Hamiltonian matrix element H_II for config s (Slater-Condon)."""
    occ = [i for i in range(norb) if (s >> i) & 1]
    e = 0.0
    for i in occ:
        e += h1[i, i]
    for ii, i in enumerate(occ):
        for j in occ[ii+1:]:
            e += eri[i, i, j, j] - eri[i, j, j, i]
    return e


def cipsi_extend(basis, ci_coeffs, norb, ne, acut, maxd, h1, eri, eps,
              use_coeffs=True, broad_frac=0.01, en_mode=False, e0=0.0):
    """Heat-bath extension with Epstein-Nesbet perturbative selection."""
    bs = set(int(x) for x in basis)

    if use_coeffs:
        if isinstance(ci_coeffs, list):
            # multi-root: use first root for imp selection fallback
            imp_ref = np.abs(ci_coeffs[0]) >= acut
            if imp_ref.sum() < 5:
                imp_ref[np.argsort(-np.abs(ci_coeffs[0]))[:5]] = True
        else:
            imp = np.abs(ci_coeffs) >= acut
            if imp.sum() < 5:
                imp[np.argsort(-np.abs(ci_coeffs))[:5]] = True
    else:
        cmax = np.max(np.abs(ci_coeffs)) if len(ci_coeffs) > 0 else 0
        threshold = cmax * broad_frac
        imp = np.abs(ci_coeffs) >= threshold
        if imp.sum() < 5:
            imp[np.argsort(-np.abs(ci_coeffs))[:min(5, len(basis))]] = True

    # === BROAD MODE ===
    if not use_coeffs:
        imp_idx = np.where(imp)[0]
        cands = set()
        for k in imp_idx:
            s = int(basis[k])
            occ = [i for i in range(norb) if (s >> i) & 1]
            vir = [i for i in range(norb) if not ((s >> i) & 1)]
            for i in occ:
                for a in vir:
                    ns = s ^ (1 << i) ^ (1 << a)
                    if ns not in bs: cands.add(ns)
            for ii, i in enumerate(occ):
                for j in occ[ii+1:]:
                    for aa, a in enumerate(vir):
                        for b in vir[aa+1:]:
                            ns = s ^ (1<<i) ^ (1<<j) ^ (1<<a) ^ (1<<b)
                            if ns in bs: continue
                            if eps > 0:
                                v = abs(eri[a,i,b,j] - eri[a,j,b,i])
                                if v < eps: continue
                            cands.add(ns)
        r = np.array(sorted(bs | cands), np.int64)
        return r[:maxd] if maxd and len(r) > maxd else r

    # === EN MODE ===
    if en_mode:
        from collections import defaultdict
        if isinstance(e0, (list, np.ndarray)):
            e0_list = list(e0)
            coeffs_list = list(ci_coeffs)
        else:
            e0_list = [e0]
            coeffs_list = [ci_coeffs]

        all_couplings = []
        for ri, (e0_r, coeffs_r) in enumerate(zip(e0_list, coeffs_list)):
            imp_r = np.abs(coeffs_r) >= acut
            if imp_r.sum() < 5:
                imp_r[np.argsort(-np.abs(coeffs_r))[:5]] = True
            imp_idx_r = np.where(imp_r)[0]

            coupling = defaultdict(float)
            for k in imp_idx_r:
                s = int(basis[k])
                ck = float(coeffs_r[k])
                occ = [i for i in range(norb) if (s >> i) & 1]
                vir = [i for i in range(norb) if not ((s >> i) & 1)]

                for i in occ:
                    for a in vir:
                        ns = s ^ (1 << i) ^ (1 << a)
                        if ns in bs: continue
                        v = h1[a, i]
                        for j_occ in occ:
                            v += eri[a, j_occ, i, j_occ] - eri[a, j_occ, j_occ, i]
                        coupling[ns] += v * ck

                for ii, i in enumerate(occ):
                    for j in occ[ii+1:]:
                        for aa, a in enumerate(vir):
                            for b in vir[aa+1:]:
                                ns = s ^ (1<<i) ^ (1<<j) ^ (1<<a) ^ (1<<b)
                                if ns in bs: continue
                                v = eri[a,i,b,j] - eri[a,j,b,i]
                                coupling[ns] += v * ck
            all_couplings.append((coupling, e0_r))

        all_candidates = set()
        for coupling, _ in all_couplings:
            all_candidates |= set(coupling.keys())

        cands = set()
        for ns in all_candidates:
            h_ii = _compute_diag(ns, norb, h1, eri)
            for coupling, e0_r in all_couplings:
                coup = coupling.get(ns, 0.0)
                if abs(coup) < 1e-15: continue
                denom = abs(e0_r - h_ii)
                if denom < 1e-12: denom = 1e-12
                delta_e = coup * coup / denom
                if delta_e > eps:
                    cands.add(ns)
                    break

        r = np.array(sorted(bs | cands), np.int64)
        return r[:maxd] if maxd and len(r) > maxd else r

    # === STANDARD PAIR-WISE MODE ===
    imp_idx = np.where(imp)[0]
    cands = []
    for k in imp_idx:
        s = int(basis[k])
        ck = abs(ci_coeffs[k])
        occ = [i for i in range(norb) if (s >> i) & 1]
        vir = [i for i in range(norb) if not ((s >> i) & 1)]
        for i in occ:
            for a in vir:
                ns = s ^ (1 << i) ^ (1 << a)
                if ns in bs: continue
                v = h1[a, i]
                for j in occ:
                    v += eri[a, j, i, j] - eri[a, j, j, i]
                if abs(v) * ck < eps: continue
                cands.append(ns)
        for ii, i in enumerate(occ):
            for j in occ[ii+1:]:
                for aa, a in enumerate(vir):
                    for b in vir[aa+1:]:
                        ns = s ^ (1<<i) ^ (1<<j) ^ (1<<a) ^ (1<<b)
                        if ns in bs: continue
                        v = abs(eri[a,i,b,j] - eri[a,j,b,i])
                        if v * ck < eps: continue
                        cands.append(ns)
    r = np.array(sorted(bs | set(cands)), np.int64)
    return r[:maxd] if maxd and len(r) > maxd else r



def build_sd_transitions(norb, nelec):
    """Build all single+double excitation operators for addon."""
    na, nb = nelec
    ops = []
    for i in range(na):
        for a in range(na, norb):
            op = np.full(2 * norb, 'I', dtype='U1')
            op[norb + a] = '+'; op[norb + i] = '-'
            ops.append(op)
    for i in range(na):
        for j in range(i + 1, na):
            for a in range(na, norb):
                for b in range(a + 1, norb):
                    op = np.full(2 * norb, 'I', dtype='U1')
                    op[norb + a] = '+'; op[norb + b] = '+'
                    op[norb + i] = '-'; op[norb + j] = '-'
                    ops.append(op)
    for i in range(nb):
        for a in range(nb, norb):
            op = np.full(2 * norb, 'I', dtype='U1')
            op[a] = '+'; op[i] = '-'
            ops.append(op)
    for i in range(nb):
        for j in range(i + 1, nb):
            for a in range(nb, norb):
                for b in range(a + 1, norb):
                    op = np.full(2 * norb, 'I', dtype='U1')
                    op[a] = '+'; op[b] = '+'
                    op[i] = '-'; op[j] = '-'
                    ops.append(op)
    return np.array(ops)

def sample_ucj_bsm(norb, nelec, t1, t2, n_shots, noise_level=0.1, seed=42):
    """Simulate QPU sampling: UCJ + depolarizing noise -> bitstring_matrix."""
    rng = np.random.default_rng(seed)
    na, nb = nelec
    t2a, t1a = safe_array(t2), safe_array(t1)
    strsa, strsb = det_strings(norb, nelec)
    n_b = len(strsb)

    probs_merged = None
    n_merged = 0
    for ts in [0.5, 1.0, 1.5, 2.0]:
        try:
            qc = QuantumCircuit(2 * norb)
            qc.append(ffsim.qiskit.PrepareHartreeFockJW(norb, nelec),
                      list(range(2 * norb)))
            qc.barrier()
            ucj = ffsim.UCJOpSpinUnbalanced.from_t_amplitudes(
                t2=(t2a*ts, t2a*ts, t2a*ts),
                t1=(t1a*ts, t1a*ts), n_reps=(8, 8))
            qc.append(ffsim.qiskit.UCJOpSpinUnbalancedJW(ucj),
                      list(range(2 * norb)))
            vec = ffsim.qiskit.final_state_vector(qc, norb=norb, nelec=nelec)
            p = np.abs(np.asarray(vec, np.complex128))**2
            if probs_merged is None:
                probs_merged = p.copy()
            else:
                probs_merged += p
            n_merged += 1
        except:
            pass
    if probs_merged is not None and n_merged > 0:
        probs_best = probs_merged / n_merged
    else:
        probs_best = np.zeros(len(strsa) * n_b); probs_best[0] = 1.0

    probs_noisy = (1 - noise_level) * probs_best + noise_level / len(probs_best)
    probs_noisy /= probs_noisy.sum()

    indices = rng.choice(len(probs_noisy), size=n_shots, p=probs_noisy)
    bsm = np.zeros((n_shots, 2 * norb), dtype=bool)
    shot_probs = np.zeros(n_shots)
    for k, idx in enumerate(indices):
        ia, ib = idx // n_b, idx % n_b
        a, b = int(strsa[ia]), int(strsb[ib])
        for i in range(norb):
            if (b >> i) & 1: bsm[k, i] = True
            if (a >> i) & 1: bsm[k, norb + i] = True
        shot_probs[k] = probs_noisy[idx]
    shot_probs /= shot_probs.sum()
    return bsm, shot_probs


# ====== SQD PIPELINE PARAMETERS ======
N_SHOTS = 100000
N_SEEDS = 1
NOISE_LEVEL = 0.1
S_CORE_ITER = 5
N_BATCHES = 10
SAMPLES_PER_BATCH = 500
MERGE_TOP_K = 1
NROOTS_INT = 20
# ======================================

# ====== CIPSI PARAMETERS ======
CIPSI_ACUT = 3e-3
CIPSI_BROAD_EPS = 5e-4
CIPSI_EN_EPS = 1e-6
CIPSI_MAX_ITER = 10
CIPSI_EXT_ROOTS = 10
CIPSI_BROAD_ITER = 2
CIPSI_BROAD_MULT = 10
CIPSI_BROAD_FRAC = 0.01
NROOTS_GROW = 20
GROW_CONV_TOL = 1e-4
GROW_MAX_CYCLE = 30
# ==================================



def label_states_fci(solver, e_list, c_list, norb, nelec, e_core):
    """Label FCI states and return (labeled_dict, s2_dict, idx_map).

    idx_map: label → index in c_list (for amplitude extraction later).
    """
    labeled, s2_dict, idx_map = {}, {}, {}
    d_cnt, q_cnt = 0, 0
    for k in range(len(e_list)):
        s2, _ = solver.spin_square(c_list[k], norb, nelec)
        s2 = float(s2)
        tag = classify_spin(s2)
        if tag == "doublet" and d_cnt < MAX_DOUBLET:
            lb = f"D{d_cnt}"; d_cnt += 1
        elif tag == "quartet" and q_cnt < MAX_QUARTET:
            lb = f"Q{q_cnt+1}"; q_cnt += 1
        else:
            continue
        labeled[lb] = float(e_core + e_list[k])
        s2_dict[lb] = s2
        idx_map[lb] = k
        if d_cnt >= MAX_DOUBLET and q_cnt >= MAX_QUARTET:
            break
    return labeled, s2_dict, idx_map


def label_states_sci(myci, e_list, c_list, norb, nelec, e_core):
    """Label SCI states and return (labeled_dict, s2_dict, idx_map)."""
    labeled, s2_dict, idx_map = {}, {}, {}
    d_cnt, q_cnt = 0, 0
    for k in range(len(e_list)):
        try:
            s2 = float(selected_ci.spin_square(c_list[k], norb, nelec)[0])
        except:
            s2 = -1.0
        tag = classify_spin(s2)
        if tag == "doublet" and d_cnt < MAX_DOUBLET:
            lb = f"D{d_cnt}"; d_cnt += 1
        elif tag == "quartet" and q_cnt < MAX_QUARTET:
            lb = f"Q{q_cnt+1}"; q_cnt += 1
        else:
            continue
        labeled[lb] = float(e_core + e_list[k])
        s2_dict[lb] = s2
        idx_map[lb] = k
        if d_cnt >= MAX_DOUBLET and q_cnt >= MAX_QUARTET:
            break
    return labeled, s2_dict, idx_map


# %% Cell 3: Build OH at given R

def build_oh(R):
    mol = gto.M(atom=f"O 0 0 0; H 0 0 {R}", basis="cc-pvdz",
                charge=0, spin=1, unit="Angstrom", verbose=0, max_memory=8000)
    mf = scf.ROHF(mol).run(conv_tol=1e-12)
    na, nb = NELEC
    mc = mcscf.CASCI(mf, ncas=NORB, nelecas=NELEC)
    mc.ncore = NCORE; mc.mo_coeff = mf.mo_coeff
    h1, e_core = mc.get_h1eff()
    eri = ao2mo.restore(1, mc.get_h2eff(), NORB)

    nocc_min, nvir = min(na, nb), NORB - max(na, nb)
    try:
        mycc = cc.CCSD(mf, frozen=NCORE)
        mycc.conv_tol = 1e-10; mycc.max_cycle = 100; mycc.kernel()
        t1_raw = mycc.t1
        t2_raw = mycc.t2
        if isinstance(t1_raw, (tuple, list)):
            t1 = safe_array(t1_raw[0][:nocc_min, :nvir])
        else:
            t1 = safe_array(t1_raw[:nocc_min, :nvir])
        if isinstance(t2_raw, (tuple, list)):
            t2 = safe_array(t2_raw[0][:nocc_min, :nocc_min, :nvir, :nvir])
        else:
            t2 = safe_array(t2_raw[:nocc_min, :nocc_min, :nvir, :nvir])
    except Exception as e:
        print(f"  CCSD fallback: {e}")
        rng = np.random.default_rng(42)
        t1 = rng.normal(0, 0.03, (nocc_min, nvir))
        t2 = rng.normal(0, 0.01, (nocc_min, nocc_min, nvir, nvir))
        t2 = t2 - t2.transpose(1, 0, 2, 3)

    return h1, eri, float(e_core), t1, t2


# %% Cell 4: Helper — compute state-resolved α/β marginals
def compute_state_marginals(ci_vec, sub_a, sub_b, full_a_map, full_b_map,
                             n_alpha_full, n_beta_full):
    """Given a CI vector on subspace (sub_a × sub_b), return α/β marginals
    mapped onto full space (0-padded where subspace missing).

    Returns:
        (p_alpha_full, p_beta_full) — numpy arrays of shape
        (n_alpha_full,), (n_beta_full,), or (None, None) on failure.
    """
    c = np.asarray(ci_vec, float)
    if c.ndim != 2:
        return None, None
    # Subspace marginals
    pa_sub = np.sum(c**2, axis=1)  # (|sub_a|,)
    pb_sub = np.sum(c**2, axis=0)  # (|sub_b|,)
    # Allocate full space
    pa_full = np.zeros(n_alpha_full)
    pb_full = np.zeros(n_beta_full)
    # Map α
    for i_sub in range(min(len(sub_a), len(pa_sub))):
        i_full = full_a_map.get(int(sub_a[i_sub]))
        if i_full is not None:
            pa_full[i_full] = pa_sub[i_sub]
    # Map β
    for i_sub in range(min(len(sub_b), len(pb_sub))):
        i_full = full_b_map.get(int(sub_b[i_sub]))
        if i_full is not None:
            pb_full[i_full] = pb_sub[i_sub]
    return pa_full, pb_full


# %% Cell 5: Compute single geometry

def compute_point(R, REF_NROOTS=15):
    print(f"\n  R = {R:.2f} A", end="", flush=True)
    t0_all = time.time()
    na, nb = NELEC

    h1, eri, e_core, t1, t2 = build_oh(R)
    strsa, strsb = det_strings(NORB, NELEC)
    n_alpha_full = len(strsa)
    n_beta_full = len(strsb)

    # Full-space string → index maps (for marginal padding)
    full_a_map = {int(s): i for i, s in enumerate(strsa)}
    full_b_map = {int(s): i for i, s in enumerate(strsb)}

    # --- Reference ---
    t0 = time.time()
    if REF_METHOD == "casci":
        mol = gto.M(atom=f"O 0 0 0; H 0 0 {R}", basis="cc-pvdz",
                    charge=0, spin=1, unit="Angstrom", verbose=0, max_memory=8000)
        mf = scf.ROHF(mol).run(conv_tol=1e-12)
        mc = mcscf.CASCI(mf, ncas=NORB, nelecas=NELEC)
        mc.ncore = NCORE; mc.mo_coeff = mf.mo_coeff
        mc.fcisolver.nroots = REF_NROOTS
        mc.fcisolver.conv_tol = 1e-10
        mc.fcisolver.max_memory = 8000
        mc.kernel()
        e_raw = mc.e_tot if hasattr(mc.e_tot, '__len__') else [mc.e_tot]
        e_fci = [float(e - e_core) for e in e_raw]
        c_raw = mc.ci if isinstance(mc.ci, (list, tuple)) else [mc.ci]
        c_fci = list(c_raw)
        fci_labeled, fci_s2, fci_idx_map = label_states_fci(
            mc.fcisolver, e_fci, c_fci, NORB, NELEC, e_core)
    else:
        solver = fci.direct_spin1.FCI()
        solver.conv_tol = 1e-10; solver.max_cycle = 200; solver.max_memory = 8000
        e_fci, c_fci = solver.kernel(h1, eri, NORB, NELEC, nroots=REF_NROOTS)
        if np.isscalar(e_fci):
            e_fci, c_fci = [float(e_fci)], [c_fci]
        fci_labeled, fci_s2, fci_idx_map = label_states_fci(
            solver, e_fci, c_fci, NORB, NELEC, e_core)
    dt_fci = time.time() - t0

    # === SQD Pipeline ===
    t0 = time.time()
    bsm_parts, probs_parts = [], []
    for s_idx in range(N_SEEDS):
        bsm_s, probs_s = sample_ucj_bsm(
            NORB, NELEC, t1, t2, N_SHOTS, NOISE_LEVEL, seed=42 + s_idx * 1000)
        bsm_parts.append(bsm_s)
        probs_parts.append(probs_s)
    bsm_raw = np.vstack(bsm_parts)
    probs_raw = np.concatenate(probs_parts)
    probs_raw /= probs_raw.sum()

    bsm_ps, probs_ps = postselect_by_hamming_right_and_left(
        bsm_raw, probs_raw, hamming_right=na, hamming_left=nb)

    for it in range(S_CORE_ITER):
        energy_sqd, sci_state, avg_occs, spin_sq = solve_fermion(
            bsm_ps, hcore=h1, eri=eri, open_shell=True)
        bsm_ps, probs_ps = recover_configurations(
            bsm_ps, probs_ps, avg_occs, na, nb, rand_seed=42 + it)

    spb = min(SAMPLES_PER_BATCH, len(bsm_ps))
    batches = subsample(bsm_ps, probs_ps, spb, N_BATCHES, rand_seed=42)
    batch_energies = []
    for i, batch in enumerate(batches):
        e_b, _, _, _ = solve_fermion(batch, hcore=h1, eri=eri, open_shell=True)
        batch_energies.append((e_b, i))
    batch_energies.sort()
    merged_rows = []
    for _, idx in batch_energies[:MERGE_TOP_K]:
        merged_rows.append(batches[idx])
    merged_batch = np.vstack(merged_rows)
    _, unique_idx = np.unique(merged_batch, axis=0, return_index=True)
    merged_batch = merged_batch[np.sort(unique_idx)]

    sqd_a, sqd_b = bitstring_matrix_to_ci_strs(merged_batch, open_shell=True)
    sqd_a = fix_ci_strs(sqd_a, NORB)
    sqd_b = fix_ci_strs(sqd_b, NORB)
    dt_sqd = time.time() - t0

    # === Ext-SQD ===
    t0 = time.time()
    t_step = time.time()
    ops = build_sd_transitions(NORB, NELEC)
    ext_bsm = enlarge_batch_from_transitions(merged_batch, ops)
    ext_a, ext_b = bitstring_matrix_to_ci_strs(ext_bsm, open_shell=True)
    ext_a = fix_ci_strs(ext_a, NORB)
    ext_b = fix_ci_strs(ext_b, NORB)
    dt_ext_enlarge = time.time() - t_step

    myci = selected_ci.SelectedCI()
    myci.conv_tol = 1e-10; myci.max_cycle = 200; myci.max_memory = 8000
    ci_ext = (ext_a, ext_b)
    nr_ext = min(NROOTS_INT, len(ext_a) * len(ext_b))
    t_step = time.time()
    el_ext, cl_ext = kernel_safe(myci, h1, eri, NORB, NELEC, ci_ext, nr_ext)
    dt_ext_diag = time.time() - t_step
    ext_labeled, ext_s2, ext_idx_map = label_states_sci(
        myci, el_ext, cl_ext, NORB, NELEC, e_core)
    dt_ext = time.time() - t0
    print(f"\n    [EXT-TIMING] enlarge={dt_ext_enlarge:.1f}s diag={dt_ext_diag:.1f}s total={dt_ext:.1f}s")

    # === CIPSI with seed diag + broad/EN selective ===
    t0 = time.time()
    seed_a = np.array(sorted(set(int(x) for x in sqd_a)), np.int64)
    seed_b = np.array(sorted(set(int(x) for x in sqd_b)), np.int64)
    basis_a = seed_a.copy()
    basis_b = seed_b.copy()
    cipsi_convergence = []
    cipsi_timings = []

    ci_seed = (seed_a, seed_b)
    t_step = time.time()
    e_seed, c_seed = kernel_safe(myci, h1, eri, NORB, NELEC, ci_seed,
                                  min(NROOTS_GROW, len(seed_a) * len(seed_b)),
                                  loose=True)
    dt_seed_diag = time.time() - t_step

    all_a = set(int(x) for x in seed_a)
    all_b = set(int(x) for x in seed_b)
    t_step = time.time()
    if e_seed:
        for cv in c_seed[:min(CIPSI_EXT_ROOTS, len(c_seed))]:
            sca = get_alpha_coeffs(cv, strsa, seed_a, ci_seed)
            ext2_a = cipsi_extend(seed_a, sca, NORB, na, CIPSI_ACUT,
                               0, h1, eri, CIPSI_BROAD_EPS * CIPSI_BROAD_MULT,
                               use_coeffs=False, broad_frac=CIPSI_BROAD_FRAC)
            all_a |= set(int(x) for x in ext2_a)
            scb = get_beta_coeffs(cv, strsb, seed_b, ci_seed)
            ext2_b = cipsi_extend(seed_b, scb, NORB, nb, CIPSI_ACUT,
                               0, h1, eri, CIPSI_BROAD_EPS * CIPSI_BROAD_MULT,
                               use_coeffs=False, broad_frac=CIPSI_BROAD_FRAC)
            all_b |= set(int(x) for x in ext2_b)
    dt_seed_broad = time.time() - t_step
    basis_a = np.array(sorted(all_a), np.int64)
    basis_b = np.array(sorted(all_b), np.int64)
    cipsi_timings.append(f"iter0: diag={dt_seed_diag:.1f}s broad={dt_seed_broad:.1f}s D=({len(basis_a)},{len(basis_b)})")
    cipsi_convergence.append({
        "iter": 0, "D_a": len(basis_a), "D_b": len(basis_b),
        "mode": "seed-broad", "energies": {}, "s2": {},
    })

    d_broad_a, d_broad_b = len(basis_a), len(basis_b)
    ci_broad = (basis_a, basis_b)
    t_step = time.time()
    nr_broad = min(NROOTS_INT, len(basis_a) * len(basis_b))
    el_broad, cl_broad = kernel_safe(myci, h1, eri, NORB, NELEC, ci_broad, nr_broad)
    broad_labeled, broad_s2, _ = label_states_sci(
        myci, el_broad, cl_broad, NORB, NELEC, e_core)
    dt_broad_diag = time.time() - t_step
    cipsi_timings.append(f"broad_diag: {dt_broad_diag:.1f}s D=({len(basis_a)},{len(basis_b)})")
    print(f"\n    [BROAD-ONLY] D=({len(basis_a)},{len(basis_b)}), precise diag: {dt_broad_diag:.1f}s")
    for lb in TARGET_STATES:
        eb = broad_labeled.get(lb)
        er = fci_labeled.get(lb)
        if eb is not None and er is not None:
            print(f"      {lb}: {eb:.10f}  ΔE(ref)={((eb-er)*1000):+.4f} mHa")

    for it in range(1, CIPSI_MAX_ITER):
        ci_s2 = (basis_a, basis_b)
        nr2 = min(NROOTS_GROW, len(basis_a) * len(basis_b))
        t_step = time.time()
        e2, c2 = kernel_safe(myci, h1, eri, NORB, NELEC, ci_s2, nr2, loose=True)
        dt_diag = time.time() - t_step
        if not e2: break

        iter_labeled, iter_s2, _ = label_states_sci(
            myci, e2, c2, NORB, NELEC, e_core)
        cipsi_convergence.append({
            "iter": it, "D_a": len(basis_a), "D_b": len(basis_b),
            "mode": "EN+broad", "energies": iter_labeled, "s2": iter_s2,
        })

        old_a, old_b = len(basis_a), len(basis_b)
        all_a = set(int(x) for x in basis_a)
        all_b = set(int(x) for x in basis_b)

        n_roots_use = min(CIPSI_EXT_ROOTS, len(c2))
        e0_list = [float(e2[ri]) for ri in range(n_roots_use)]

        t_step = time.time()
        ac_list = [get_alpha_coeffs(c2[ri], strsa, basis_a, ci_s2)
                   for ri in range(n_roots_use)]
        ext2_a = cipsi_extend(basis_a, ac_list, NORB, na, CIPSI_ACUT,
                           0, h1, eri, CIPSI_EN_EPS, use_coeffs=True,
                           broad_frac=CIPSI_BROAD_FRAC, en_mode=True, e0=e0_list)
        all_a |= set(int(x) for x in ext2_a)

        bc_list = [get_beta_coeffs(c2[ri], strsb, basis_b, ci_s2)
                   for ri in range(n_roots_use)]
        ext2_b = cipsi_extend(basis_b, bc_list, NORB, nb, CIPSI_ACUT,
                           0, h1, eri, CIPSI_EN_EPS, use_coeffs=True,
                           broad_frac=CIPSI_BROAD_FRAC, en_mode=True, e0=e0_list)
        all_b |= set(int(x) for x in ext2_b)
        dt_en = time.time() - t_step

        t_step = time.time()
        max_coeff_a = np.zeros(len(basis_a))
        max_coeff_b = np.zeros(len(basis_b))
        for ri in range(n_roots_use):
            ac = get_alpha_coeffs(c2[ri], strsa, basis_a, ci_s2)
            max_coeff_a = np.maximum(max_coeff_a, np.abs(ac))
            bc = get_beta_coeffs(c2[ri], strsb, basis_b, ci_s2)
            max_coeff_b = np.maximum(max_coeff_b, np.abs(bc))

        broad_top_n = max(10, int(len(basis_a) * CIPSI_BROAD_FRAC))
        top_a_idx = np.argsort(-max_coeff_a)[:broad_top_n]
        top_a = basis_a[top_a_idx]
        for s in top_a:
            s_int = int(s)
            occ = [i for i in range(NORB) if (s_int >> i) & 1]
            vir = [i for i in range(NORB) if not ((s_int >> i) & 1)]
            for i in occ:
                for a in vir:
                    all_a.add(s_int ^ (1 << i) ^ (1 << a))

        broad_top_n_b = max(10, int(len(basis_b) * CIPSI_BROAD_FRAC))
        top_b_idx = np.argsort(-max_coeff_b)[:broad_top_n_b]
        top_b = basis_b[top_b_idx]
        for s in top_b:
            s_int = int(s)
            occ = [i for i in range(NORB) if (s_int >> i) & 1]
            vir = [i for i in range(NORB) if not ((s_int >> i) & 1)]
            for i in occ:
                for a in vir:
                    all_b.add(s_int ^ (1 << i) ^ (1 << a))
        dt_broad = time.time() - t_step

        n_add_a = len(all_a) - old_a
        n_add_b = len(all_b) - old_b
        basis_a = np.array(sorted(all_a), np.int64)
        basis_b = np.array(sorted(all_b), np.int64)
        cipsi_timings.append(f"iter{it}: diag={dt_diag:.1f}s EN={dt_en:.1f}s broad={dt_broad:.1f}s +({n_add_a},{n_add_b}) D=({len(basis_a)},{len(basis_b)})")
        if len(basis_a) == old_a and len(basis_b) == old_b: break

    ci_cipsi = (basis_a, basis_b)
    nr_cipsi = min(NROOTS_INT, len(basis_a) * len(basis_b))
    t_step = time.time()
    el_cipsi, cl_cipsi = kernel_safe(myci, h1, eri, NORB, NELEC, ci_cipsi, nr_cipsi)
    dt_final = time.time() - t_step
    cipsi_labeled, cipsi_s2, cipsi_idx_map = label_states_sci(
        myci, el_cipsi, cl_cipsi, NORB, NELEC, e_core)
    dt_cipsi = time.time() - t0

    cipsi_timings.append(f"final: diag={dt_final:.1f}s D=({len(basis_a)},{len(basis_b)})")
    print(f"\n    [CIPSI-TIMING] " + " | ".join(cipsi_timings))

    print(f"    [EN-IMPROVE] Broad D=({d_broad_a},{d_broad_b}) → Final D=({len(basis_a)},{len(basis_b)})")
    for lb in TARGET_STATES:
        eb = broad_labeled.get(lb)
        ef = cipsi_labeled.get(lb)
        er = fci_labeled.get(lb)
        if eb is not None and ef is not None and er is not None:
            de_broad = (eb - er) * 1000
            de_final = (ef - er) * 1000
            improve = de_broad - de_final
            print(f"      {lb}: broad={de_broad:+.4f} → final={de_final:+.4f} mHa "
                  f"(EN improved {improve:.4f} mHa)")

    cipsi_convergence.append({
        "iter": "final", "D_a": len(basis_a), "D_b": len(basis_b),
        "mode": "final", "energies": cipsi_labeled, "s2": cipsi_s2,
    })

    # === Natural orbital occupation analysis (GS only, as before) ===
    nat_occ = {}
    try:
        if len(c_fci) > 0:
            rdm1_ref = fci.direct_spin1.make_rdm1(c_fci[0], NORB, NELEC)
            occ_ref = np.sort(np.linalg.eigvalsh(rdm1_ref))[::-1]
            nat_occ["ref"] = occ_ref.tolist()

        if len(cl_ext) > 0:
            myci._strs = ci_ext
            rdm1_ext = myci.make_rdm1(cl_ext[0], NORB, NELEC)
            occ_ext = np.sort(np.linalg.eigvalsh(rdm1_ext))[::-1]
            nat_occ["ext"] = occ_ext.tolist()

        if len(cl_cipsi) > 0:
            myci._strs = ci_cipsi
            rdm1_cipsi = myci.make_rdm1(cl_cipsi[0], NORB, NELEC)
            occ_cipsi = np.sort(np.linalg.eigvalsh(rdm1_cipsi))[::-1]
            nat_occ["cipsi"] = occ_cipsi.tolist()

        if nat_occ.get("ref"):
            occ_str = " ".join(f"{o:.4f}" for o in nat_occ["ref"])
            print(f"\n    [NO] Ref:  {occ_str}")
        if nat_occ.get("ext") and nat_occ.get("ref"):
            diff = np.max(np.abs(np.array(nat_occ["ext"]) - np.array(nat_occ["ref"])))
            print(f"    [NO] Ext:  max|Δn|={diff:.6f}")
        if nat_occ.get("cipsi") and nat_occ.get("ref"):
            diff = np.max(np.abs(np.array(nat_occ["cipsi"]) - np.array(nat_occ["ref"])))
            print(f"    [NO] CIPSI:   max|Δn|={diff:.6f}")
    except Exception as e:
        print(f"\n    [NO] Failed: {e}")

    # === Diagnostics ===
    ext_set_a = set(int(x) for x in ext_a)
    cipsi_set_a = set(int(x) for x in basis_a)
    missing_a = sorted(ext_set_a - cipsi_set_a)

    diag = {
        "n_missing_a": len(missing_a),
        "n_ext_a": len(ext_set_a), "n_cipsi_a": len(cipsi_set_a),
        "n_ext_b": len(ext_b), "n_cipsi_b": len(basis_b),
    }

    # WF analysis using Ext-SQD CI vector (unchanged)
    wf_analysis = {}
    if len(cl_ext) > 0:
        cr_ext = np.asarray(cl_ext[0], float)
        if cr_ext.ndim == 2:
            ext_str_map_a = {int(s): i for i, s in enumerate(ext_a)}
            alpha_weights = {}
            for ia, s in enumerate(ext_a):
                if ia < cr_ext.shape[0]:
                    alpha_weights[int(s)] = float(np.sum(cr_ext[ia, :]**2))
            total_weight = sum(alpha_weights.values())
            cipsi_weight = sum(alpha_weights.get(s, 0) for s in cipsi_set_a)
            ext_only_s = ext_set_a - cipsi_set_a
            ext_only_w = sum(alpha_weights.get(s, 0) for s in ext_only_s)
            ext_only_max = max((alpha_weights.get(s, 0) for s in ext_only_s), default=0)

            wf_analysis = {
                "ext_recovery_pct": 100.0 * total_weight / max(total_weight, 1e-20),
                "cipsi_recovery_pct": 100.0 * cipsi_weight / max(total_weight, 1e-20),
                "ext_per_config": total_weight / max(len(ext_set_a), 1),
                "cipsi_per_config": cipsi_weight / max(len(cipsi_set_a), 1),
                "categories": {
                    "ext_only": {"n": len(ext_only_s),
                                 "pct": 100.0 * ext_only_w / max(total_weight, 1e-20),
                                 "max": ext_only_max},
                },
            }
            if wf_analysis["ext_per_config"] > 0:
                wf_analysis["efficiency_ratio"] = (
                    wf_analysis["cipsi_per_config"] / wf_analysis["ext_per_config"])
            avg_ext_only = ext_only_w / max(len(ext_only_s), 1)
            if avg_ext_only > 0:
                wf_analysis["selectivity_ratio"] = (
                    wf_analysis["cipsi_per_config"] / avg_ext_only)

            def cum_curve(config_set, weights, max_pts=100):
                ws = sorted([weights.get(s, 0) for s in config_set], reverse=True)
                cum = np.cumsum(ws)
                if len(cum) == 0: return [], []
                cum_pct = (cum / max(total_weight, 1e-20) * 100).tolist()
                if len(cum_pct) <= max_pts:
                    return list(range(1, len(cum_pct) + 1)), cum_pct
                step = max(1, len(cum_pct) // max_pts)
                idx = list(range(0, len(cum_pct), step))
                if idx[-1] != len(cum_pct) - 1:
                    idx.append(len(cum_pct) - 1)
                return [i + 1 for i in idx], [cum_pct[i] for i in idx]

            full_set_a = set(int(s) for s in strsa)
            opt_x, opt_y = cum_curve(full_set_a, alpha_weights)
            ext_x, ext_y = cum_curve(ext_set_a, alpha_weights)
            cipsi_x, cipsi_y = cum_curve(cipsi_set_a, alpha_weights)
            wf_analysis["opt_curve"] = {"x": opt_x, "y": opt_y}
            wf_analysis["ext_curve"] = {"x": ext_x, "y": ext_y}
            wf_analysis["cipsi_curve"] = {"x": cipsi_x, "y": cipsi_y}

    # ================================================================
    # === STATE-RESOLVED α/β MARGINALS (new)                      ===
    # ================================================================
    # wf_amps[state_label] = {
    #   "ref_a": [n_alpha_full floats],  "ref_b": [n_beta_full floats],
    #   "ext_a": [...],                   "ext_b": [...],
    #   "cipsi_a": [...],                 "cipsi_b": [...],
    # }
    # All arrays are on the FULL space (0-padded where subspace missing),
    # so they can be directly compared across methods/states.
    wf_amps = {}
    try:
        for lb in TARGET_STATES:
            if lb not in fci_labeled:
                continue
            entry = {}

            # --- Reference (FCI): already on full space ---
            fi = fci_idx_map.get(lb)
            if fi is not None and fi < len(c_fci):
                c_r = np.asarray(c_fci[fi], float)
                if c_r.ndim == 2:
                    pa_r = np.sum(c_r**2, axis=1)
                    pb_r = np.sum(c_r**2, axis=0)
                    # FCI already on full space — verify shape
                    if len(pa_r) == n_alpha_full and len(pb_r) == n_beta_full:
                        entry["ref_a"] = pa_r.tolist()
                        entry["ref_b"] = pb_r.tolist()
                    else:
                        # Shape mismatch: pad manually if needed
                        pa_full = np.zeros(n_alpha_full)
                        pb_full = np.zeros(n_beta_full)
                        pa_full[:len(pa_r)] = pa_r
                        pb_full[:len(pb_r)] = pb_r
                        entry["ref_a"] = pa_full.tolist()
                        entry["ref_b"] = pb_full.tolist()

            # --- Ext-SQD: subspace (ext_a, ext_b) → pad to full ---
            ei = ext_idx_map.get(lb)
            if ei is not None and ei < len(cl_ext):
                pa_e, pb_e = compute_state_marginals(
                    cl_ext[ei], ext_a, ext_b,
                    full_a_map, full_b_map, n_alpha_full, n_beta_full)
                if pa_e is not None:
                    entry["ext_a"] = pa_e.tolist()
                    entry["ext_b"] = pb_e.tolist()

            # --- CIPSI: subspace (basis_a, basis_b) → pad to full ---
            ci_i = cipsi_idx_map.get(lb)
            if ci_i is not None and ci_i < len(cl_cipsi):
                pa_c, pb_c = compute_state_marginals(
                    cl_cipsi[ci_i], basis_a, basis_b,
                    full_a_map, full_b_map, n_alpha_full, n_beta_full)
                if pa_c is not None:
                    entry["cipsi_a"] = pa_c.tolist()
                    entry["cipsi_b"] = pb_c.tolist()

            if entry:
                wf_amps[lb] = entry

        if wf_amps:
            states_stored = list(wf_amps.keys())
            per_state_keys = {lb: list(wf_amps[lb].keys()) for lb in states_stored}
            print(f"\n    [AMP] States stored: {states_stored}")
            print(f"    [AMP] Full α-space: {n_alpha_full}, β-space: {n_beta_full}")
            print(f"    [AMP] Per-state keys: {per_state_keys}")
    except Exception as e:
        import traceback
        print(f"\n    [AMP] Failed: {e}")
        traceback.print_exc()

    wf_amps_meta = {
        "n_alpha_full": n_alpha_full,
        "n_beta_full": n_beta_full,
        "states_stored": list(wf_amps.keys()),
    }

    # Important missing analysis
    n_important = 0
    max_missing_w = 0.0
    if len(cl_ext) > 0 and cr_ext.ndim == 2:
        for s in missing_a:
            ia = ext_str_map_a.get(s)
            if ia is not None and ia < cr_ext.shape[0]:
                w = float(np.max(np.abs(cr_ext[ia, :])))
                max_missing_w = max(max_missing_w, w)
                if w > 0.01: n_important += 1
    diag["n_important_missing"] = n_important
    diag["max_missing_weight"] = max_missing_w

    dim_ext = max(len(ext_a), len(ext_b))
    dim_cipsi = max(len(basis_a), len(basis_b))
    if len(missing_a) > 0:
        print(f"\n    [DIAG] Missing alpha: {len(missing_a)}/{len(ext_set_a)} "
              f"(important: {n_important})")
    conv_str = " → ".join(f"({c['D_a']},{c['D_b']})" for c in cipsi_convergence)
    print(f"\n    [CONV] {conv_str}")
    if wf_analysis:
        eo = wf_analysis.get("categories", {}).get("ext_only", {})
        print(f"    [WF] Ext: {wf_analysis['ext_recovery_pct']:.2f}% | "
              f"CIPSI: {wf_analysis['cipsi_recovery_pct']:.2f}% | "
              f"eff: {wf_analysis.get('efficiency_ratio', 0):.1f}x | "
              f"ext-only: {eo.get('n', 0)} ({eo.get('pct', 0):.4f}%)")

    print(f"  Ref:{dt_fci:.0f}s  SQD({len(sqd_a)},{len(sqd_b)}):{dt_sqd:.0f}s  "
          f"Ext({len(ext_a)},{len(ext_b)}):{dt_ext:.0f}s  "
          f"CIPSI({len(basis_a)},{len(basis_b)}):{dt_cipsi:.0f}s  "
          f"[{time.time()-t0_all:.0f}s]", flush=True)

    return {
        "R": R,
        "fci": fci_labeled,
        "ext_sqd": ext_labeled,
        "cipsi": cipsi_labeled,
        "fci_s2": fci_s2,
        "ext_s2": ext_s2,
        "cipsi_s2": cipsi_s2,
        "dims": {"sqd_a": len(sqd_a), "sqd_b": len(sqd_b),
                 "ext_a": len(ext_a), "ext_b": len(ext_b),
                 "ext": dim_ext, "cipsi": dim_cipsi,
                 "cipsi_a": len(basis_a), "cipsi_b": len(basis_b),
                 "shots": N_SHOTS * N_SEEDS},
        "diag": diag,
        "cipsi_convergence": cipsi_convergence,
        "wf_analysis": wf_analysis,
        "nat_occ": nat_occ,
        "wf_amps": wf_amps,
        "wf_amps_meta": wf_amps_meta,
    }


# %% Cell 6: Run PES scan (incremental save + resume)


R_VALUES = [0.70, 0.75, 0.80, 0.85, 0.90, 0.95, 1.00, 1.03,
            1.10, 1.15, 1.20, 1.25, 1.30, 1.35, 1.40, 1.45,
            1.50, 1.55, 1.60, 1.65, 1.70, 1.75, 1.80, 1.90,
            2.00, 2.10, 2.20, 2.30, 2.40, 2.50, 2.60, 2.70,
            2.80, 2.90, 3.00]
REF_METHOD = "fci"
REF_NROOTS = 20

out_fname = f"oh_{NORB}o_pes.json"

print(f"OH radical (7e,{NORB}o)/cc-pVDZ PES scan — STATE-RESOLVED AMPLITUDE version")
print(f"R = {R_VALUES}  ({len(R_VALUES)} points)")
print(f"Ref = {REF_METHOD} ({REF_NROOTS} roots)")
print(f"Output: {out_fname}  (fresh run, will overwrite if exists)")

strsa_full, strsb_full = det_strings(NORB, NELEC)
print(f"Full space: {len(strsa_full)} alpha x {len(strsb_full)} beta "
      f"= {len(strsa_full)*len(strsb_full):,} det")

t_total = time.time()
all_data = []

for R in R_VALUES:
    pt = compute_point(R, REF_NROOTS)
    all_data.append(pt)

    # === incremental save after every R point (atomic write) ===
    try:
        tmp_fname = out_fname + ".tmp"
        with open(tmp_fname, "w") as f:
            json.dump(all_data, f, indent=2)
        os.replace(tmp_fname, out_fname)  # atomic swap
        print(f"    [SAVED] {out_fname}  ({len(all_data)}/{len(R_VALUES)} pts)",
              flush=True)
    except Exception as e:
        print(f"    [SAVE FAILED] {e}", flush=True)

print(f"\nTotal PES time: {time.time()-t_total:.0f}s")
print(f"[final saved] {out_fname}  ({len(all_data)} points)")


# %% Cell 7: Collect states & summary

all_labels = set()
for pt in all_data:
    all_labels |= set(pt["fci"].keys())
STATES = [lb for lb in TARGET_STATES if lb in all_labels]
print(f"States for plotting: {STATES}")

print(f"\n{'R':>5} |", end="")
for lb in STATES:
    print(f" {lb+' FCI':>12} {lb+' Ext':>10} {lb+' CIPSI':>10} |", end="")
print()
print("-" * (7 + len(STATES) * 35))

for pt in all_data:
    R = pt["R"]
    row = f"{R:5.2f} |"
    for lb in STATES:
        ef = pt["fci"].get(lb)
        ee = pt["ext_sqd"].get(lb)
        eh = pt["cipsi"].get(lb)
        row += f" {ef:12.6f}" if ef else f" {'':>12}"
        if ee and ef:
            row += f" {(ee-ef)*1000:+9.2f}"
        else:
            row += f" {'MISS':>10}"
        if eh and ef:
            row += f" {(eh-ef)*1000:+9.2f}"
        else:
            row += f" {'MISS':>10}"
        row += " |"
    print(row)

print("\nDone. State-resolved amplitudes stored in wf_amps[state_label].")
print("Use oh_figures.py (new version) to generate state-by-state plots.")

PySCF threads: 24
OH radical (7e,18o)/cc-pVDZ PES scan — STATE-RESOLVED AMPLITUDE version
R = [0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0, 1.03, 1.1, 1.15, 1.2, 1.25, 1.3, 1.35, 1.4, 1.45, 1.5, 1.55, 1.6, 1.65, 1.7, 1.75, 1.8, 1.9, 2.0, 2.1, 2.2, 2.3, 2.4, 2.5, 2.6, 2.7, 2.8, 2.9, 3.0]  (35 points)
Ref = fci (20 roots)
Output: oh_18o_pes.json  (fresh run, will overwrite if exists)
Full space: 3060 alpha x 816 beta = 2,496,960 det

  R = 0.70 A
    [EXT-TIMING] enlarge=1.7s diag=358.8s total=360.9s

    [BROAD-ONLY] D=(1400,806), precise diag: 157.2s
      D0: -75.4158912562  ΔE(ref)=+0.3813 mHa
      D1: -75.4158652737  ΔE(ref)=+0.3939 mHa
      D2: -75.2430797912  ΔE(ref)=+0.3877 mHa
      Q1: -74.3952984912  ΔE(ref)=+658.5400 mHa

    [CIPSI-TIMING] iter0: diag=4.6s broad=0.1s D=(1400,806) | broad_diag: 157.2s D=(1400,806) | iter1: diag=64.4s EN=0.3s broad=0.1s +(228,4) D=(1628,810) | iter2: diag=82.1s EN=0.4s broad=0.1s +(53,0) D=(1681,810) | iter3: diag=85.7s EN=0.3s broad=0.1s +(0,0) D=